# LOAD LIBRARY

In [10]:
# Library preprocessing text
import pandas as pd
from datetime import datetime
import os
import re

# Library bar chart dan wordcloud
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from wordcloud import WordCloud, STOPWORDS
from collections import Counter

# Library stemming
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Library Gemini (analisis sentimen)
from openai import OpenAI
from tqdm import tqdm
import time

# PREPROCESSING

In [ ]:
project_name = "bemui"
platform = "threads"  # 'x', 'ig', 'threads', 'fb', 'article'

# ======================================================
# AUTO-RESOLVE PATHS DAN KOLOM
# ======================================================
PLATFORM_CONFIG = {
    'x': {'folder': 'x/tweets', 'suffix': 'tweets', 'id_col': 'tweet_id_str', 'label': 'Tweet', 'date_col': 'created_at'},
    'ig': {'folder': 'ig/posts', 'suffix': 'ig-posts', 'id_col': 'post_id_str', 'label': 'Post', 'date_col': 'created_at'},
    'threads': {'folder': 'threads/posts', 'suffix': 'threads-posts', 'id_col': 'post_id_str', 'label': 'Post', 'date_col': 'created_at'},
    'fb': {'folder': 'fb/posts', 'suffix': 'fb-posts', 'id_col': 'post_id_str', 'label': 'Post', 'date_col': 'created_at'},
    'article': {'folder': 'article', 'suffix': 'articles', 'id_col': 'article_id_str', 'label': 'Article', 'date_col': 'published_at'},
}

config = PLATFORM_CONFIG[platform]
file_name = f"{project_name}-{config['suffix']}.csv"
id_col = config['id_col']
data_label = config['label']
date_col = config['date_col']

# SETUP FOLDER
base_folder = f"../../result/{project_name}/{config['folder']}"
output_folder = f"{base_folder}/preprocessing_ai"
os.makedirs(output_folder, exist_ok=True)

data = pd.read_csv(f"{base_folder}/{file_name}")

# Konversi kolom tanggal ke datetime aware dan ubah ke zona waktu Asia/Jakarta
data[date_col] = pd.to_datetime(
    data[date_col], format='%a %b %d %H:%M:%S %z %Y'
).dt.tz_convert('Asia/Jakarta').dt.strftime('%a %b %d %H:%M:%S %z %Y')

# Cek kolom tanggal & waktu ada. Jika belum, buat.
if 'tanggal' not in data.columns or 'waktu' not in data.columns:
    data['tanggal'] = pd.to_datetime(
        data[date_col], format='%a %b %d %H:%M:%S %z %Y'
    ).dt.strftime('%Y-%m-%d')

    data['waktu'] = pd.to_datetime(
        data[date_col], format='%a %b %d %H:%M:%S %z %Y'
    ).dt.strftime('%H:%M:%S')

# === ARTICLE: gabung teks untuk sentiment ===
if platform == 'article':
    def _build_sentiment_text(row):
        title = str(row.get('title', '')).strip()
        ft = str(row.get('full_text', '')).strip()
        desc = str(row.get('description', '')).strip()
        if title in ('-', ''): title = ''
        if ft in ('-', ''): ft = ''
        if desc in ('-', ''): desc = ''
        body = ft or desc
        if not body:
            return ''
        return f"{title}. {body}" if title else body

    data['_sentiment_text'] = data.apply(_build_sentiment_text, axis=1)
    before_filter = len(data)
    data = data[data['_sentiment_text'].str.strip() != ''].reset_index(drop=True)
    print(f"Filter artikel tidak valid (tanpa description & full_text): {before_filter} → {len(data)} ({before_filter - len(data)} removed)")

# Dynamic column: pertahankan semua kolom asli, full_text di akhir
cols = [c for c in data.columns if c != 'full_text'] + ['full_text']
df = data[cols].copy()

df.drop_duplicates(subset=id_col, keep='last', inplace=True)

if platform != 'article':
    df = df[df['full_text'].notna() & (df['full_text'].str.strip() != "-") & (df['full_text'].str.strip() != "")]

df.info()
df.head(5)

In [ ]:
# CASE FOLDING
import unicodedata

def case_folding(tweet):
  if isinstance(tweet, str):
    tweet = unicodedata.normalize('NFKD', tweet)  # Unicode fancy → ASCII (𝗦𝗮𝘁𝗴𝗮𝘀 → Satgas)
    lowerCase = tweet.lower()
    return lowerCase

text_source = '_sentiment_text' if platform == 'article' else 'full_text'
df['case_folding'] = df[text_source].apply(case_folding)

df.head(5)

In [13]:
# NORMALIZE
def normalize(text, kamus):
    import re

    if not isinstance(text, str):
        return "", [], [], []

    # Protect @mentions dan #hashtags dari kamus
    protected = {}
    prot_counter = [0]
    def protect(match):
        key = f"__PROT{prot_counter[0]}__"
        protected[key] = match.group(0)
        prot_counter[0] += 1
        return f" {key} "

    text = re.sub(r'@[^\s]+', protect, text)
    text = re.sub(r'#[^\s]+', protect, text)

    # Cleansing ringan
    text = re.sub(r'^rt[\s]+', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r'', text)
    text = re.sub(r'&gt;', ' ', text)
    text = re.sub(r'&amp;|&', ' ', text)
    text = re.sub(r'amp[\s]+', ' ', text)
    text = re.sub(r'[^\w\s:%$.\-,!?/\"\\\']', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # Normalize slang words
    kata_baku = []
    kata_diganti = []
    kata_tidak_baku_hash = []
    sorted_patterns = sorted(kamus.keys(), key=len, reverse=True)

    for slang in sorted_patterns:
        pattern = re.compile(rf'\b{re.escape(slang)}\b')
        matches = pattern.findall(text)
        if matches:
            for match in matches:
                kata_diganti.append(match)
                kata_tidak_baku_hash.append(match)
            text = pattern.sub(kamus[slang], text)
            kata_baku.extend([kamus[slang]] * len(matches))

    # Restore protected @mentions dan #hashtags
    for key, val in protected.items():
        text = text.replace(key, val)

    text = re.sub(r'\s+', ' ', text).strip()

    return text, kata_baku, kata_diganti, kata_tidak_baku_hash


# Load kamus
kamus_kata = pd.read_csv("https://raw.githubusercontent.com/whdkelilauw/myfiles/main/kamus_normalize.csv")
kamus_kata['slang'] = kamus_kata['slang'].str.strip()
kamus_kata['formal'] = kamus_kata['formal'].str.strip()
kamus = dict(zip(kamus_kata['slang'], kamus_kata['formal']))

df['normalize'] = df['case_folding'].apply(lambda x: normalize(x, kamus)[0])

df.head(5)

,post_id_str,created_at,user_id_str,username,name,user_mentions,hashtags,like_count,reply_count,repost_count,...,media_url,post_url,relation_type,target_post_id_str,target_username,tanggal,waktu,full_text,case_folding,normalize
0,3966562494574561312,Tue Aug 18 23:38:38 +0700 2026,63081966924,jaka_hb,jaka HB,-,-,0,0,0,...,-,https://www.threads.com/@jaka_hb/post/DcMD5K0gQgg,-,-,-,2026-08-18,23:38:38,Nih\n\nhttps://jakarta.tribunnews.com/jakarta/...,nih\n\nhttps://jakarta.tribunnews.com/jakarta/...,nih
1,3966562240602212698,Tue Aug 18 23:38:08 +0700 2026,69220082837,eexi,Rezky Fauziah,-,-,0,0,0,...,-,https://www.threads.com/@eexi/post/DcMD1eSkola,replied,3966468273739127284,adiitms,2026-08-18,23:38:08,Malah mikir cosplay jadi org biasa aja dulu pa...,malah mikir cosplay jadi org biasa aja dulu pa...,malah mikir cosplay jadi orang biasa saja dulu...
2,3966560917222668562,Tue Aug 18 23:38:08 +0700 2026,47722815717,dark_lyre,Xuě Láng,-,-,0,0,0,...,https://instagram.fjog6-1.fna.fbcdn.net/v/t51....,https://www.threads.com/@dark_lyre/post/DcMDiN...,replied,3966308848940057222,koalisi.indonesia.maju,2026-08-18,23:38:08,"Kak boleh numpang tanya, itu ngapain polwan pa...","kak boleh numpang tanya, itu ngapain polwan pa...","kak boleh numpang tanya, itu mengapa polwan pa..."
3,3966562158544928008,Tue Aug 18 23:37:58 +0700 2026,76189409752,mnbvcxzdfgjytr,.,-,-,0,0,0,...,-,https://www.threads.com/@mnbvcxzdfgjytr/post/D...,replied,3966535614842500820,johannesclintons,2026-08-18,23:37:58,"gini ya, hampir semua gedung di thamrin dekat ...","gini ya, hampir semua gedung di thamrin dekat ...","begini ya, hampir semua gedung di thamrin deka..."
4,3966561721187370588,Tue Aug 18 23:37:08 +0700 2026,67943823428,imam_munthasir,Bang Bewok,-,#JakartaTerkini;#DemoMahasiswa;#bundaranhi;#be...,0,0,0,...,https://instagram.fjog6-1.fna.fbcdn.net/v/t51....,https://www.threads.com/@imam_munthasir/post/D...,-,-,-,2026-08-18,23:37:08,18 Agustus 2026. Aksi unjuk rasa mahasiswa UI ...,18 agustus 2026. aksi unjuk rasa mahasiswa ui ...,18 agustus 2026. aksi unjuk rasa mahasiswa ui ...


In [14]:
len(df)

16700

In [15]:
df_normalize = df.copy()

In [17]:
# === FILTER SPAM/IKLAN ===
SPAM_TERMS = [
    'gofood', 'go food', 'grabfood', 'grab food',
    'shopeefood', 'shopee food', 'sfood',
    'daget', 'voucher', 'promo', 'diskon'
]

SPAM_CONTACT = [
    'hub. kami', 'hub kami', 'hubungi kami'
]

def is_spam(text_normalize, text_casefold):
    if not isinstance(text_normalize, str) or not isinstance(text_casefold, str):
        return False
    for phrase in SPAM_CONTACT:
        if phrase in text_casefold:
            return True
    count = 0
    for term in SPAM_TERMS:
        if term in text_normalize:
            count += 1
    if re.search(r'08\d{8,12}', text_normalize):
        count += 1
    return count >= 3

before_spam = len(df)
spam_mask = df.apply(lambda r: is_spam(r['normalize'], r['case_folding']), axis=1)

df_spam = df[spam_mask][['full_text']].reset_index(drop=True)
df_spam.to_csv(f"{output_folder}/{project_name}_spam_removed.csv", index=False)
print(f"[OK] Spam tersimpan → {output_folder}/{project_name}_spam_removed.csv ({len(df_spam)} {data_label.lower()}s)")

df = df[~spam_mask].reset_index(drop=True)
print(f"Filter spam/iklan: {before_spam} → {len(df)} ({before_spam - len(df)} removed)")

# === FILTER KEYWORD CRAWLING (cek dari normalize) ===
CRAWL_KEYWORDS = [
    "bem ui", "bem", "demo", "aksi", "unjuk", "sudirman", "dukuh atas", "reynold", "fatimah", "bundaran",
    "merebutkemerdekaan", "mahasiswa", "stay safe", "anarkis", "perjuangan"
]

before_kw = len(df)
keyword_mask = df['normalize'].apply(
    lambda x: any(kw in x for kw in CRAWL_KEYWORDS) if isinstance(x, str) else False
)

df_no_keyword = df[~keyword_mask][['full_text']].reset_index(drop=True)
df_no_keyword.to_csv(f"{output_folder}/{project_name}_no_keyword_removed.csv", index=False)
print(f"[OK] No-keyword tersimpan → {output_folder}/{project_name}_no_keyword_removed.csv ({len(df_no_keyword)} {data_label.lower()}s)")

df = df[keyword_mask].reset_index(drop=True)
print(f"Filter keyword: {before_kw} → {len(df)} ({before_kw - len(df)} removed)")

# CLEANSING
def cleansing(tweet):
  if not isinstance(tweet, str):
      return ''
  tweet = tweet.encode('ascii', 'replace').decode('ascii')
  tweet = re.sub(r'\bb\d{1,3}\b|\d+', lambda m: m.group() if m.group()[0].isalpha() else '', tweet, flags=re.IGNORECASE)
  tweet = re.sub(r'^rt[\s]+', ' ', tweet)
  tweet = re.sub(r'@[^\s]+', ' ', tweet)
  tweet = re.sub(r'#[^\s]+', ' ', tweet)
  tweet = re.sub(r'http\S+|www\.\S+', '', tweet)
  emoji_pattern = re.compile(
      "["
      "\U0001F600-\U0001F64F"
      "\U0001F300-\U0001F5FF"
      "\U0001F680-\U0001F6FF"
      "\U0001F1E0-\U0001F1FF"
      "\U00002702-\U000027B0"
      "\U000024C2-\U0001F251"
      "]+", flags=re.UNICODE)
  tweet = emoji_pattern.sub(r'', tweet)
  tweet = re.sub(r'[^\w\s]', ' ', tweet)
  tweet = re.sub(r'&amp;|&', ' ', tweet)
  tweet = re.sub(r'amp[\s]+', ' ', tweet)
  tweet = re.sub(r'\s+', ' ', tweet).strip()
  return tweet

df['cleansing'] = df['normalize'].apply(cleansing)

df = df[df['cleansing'].notna() & (df['cleansing'].str.strip() != "-") & (df['cleansing'].str.strip() != "")]

df.head(5)

[OK] Spam tersimpan → ../../result/bemui/threads/posts/preprocessing_gemini/bemui_spam_removed.csv (0 posts)
Filter spam/iklan: 11108 → 11108 (0 removed)
[OK] No-keyword tersimpan → ../../result/bemui/threads/posts/preprocessing_gemini/bemui_no_keyword_removed.csv (0 posts)
Filter keyword: 11108 → 11108 (0 removed)


,post_id_str,created_at,user_id_str,username,name,user_mentions,hashtags,like_count,reply_count,repost_count,...,post_url,relation_type,target_post_id_str,target_username,tanggal,waktu,full_text,case_folding,normalize,cleansing
0,3966562240602212698,Tue Aug 18 23:38:08 +0700 2026,69220082837,eexi,Rezky Fauziah,-,-,0,0,0,...,https://www.threads.com/@eexi/post/DcMD1eSkola,replied,3966468273739127284,adiitms,2026-08-18,23:38:08,Malah mikir cosplay jadi org biasa aja dulu pa...,malah mikir cosplay jadi org biasa aja dulu pa...,malah mikir cosplay jadi orang biasa saja dulu...,malah mikir cosplay jadi orang biasa saja dulu...
1,3966560917222668562,Tue Aug 18 23:38:08 +0700 2026,47722815717,dark_lyre,Xuě Láng,-,-,0,0,0,...,https://www.threads.com/@dark_lyre/post/DcMDiN...,replied,3966308848940057222,koalisi.indonesia.maju,2026-08-18,23:38:08,"Kak boleh numpang tanya, itu ngapain polwan pa...","kak boleh numpang tanya, itu ngapain polwan pa...","kak boleh numpang tanya, itu mengapa polwan pa...",kak boleh numpang tanya itu mengapa polwan pak...
2,3966562158544928008,Tue Aug 18 23:37:58 +0700 2026,76189409752,mnbvcxzdfgjytr,.,-,-,0,0,0,...,https://www.threads.com/@mnbvcxzdfgjytr/post/D...,replied,3966535614842500820,johannesclintons,2026-08-18,23:37:58,"gini ya, hampir semua gedung di thamrin dekat ...","gini ya, hampir semua gedung di thamrin dekat ...","begini ya, hampir semua gedung di thamrin deka...",begini ya hampir semua gedung di thamrin dekat...
3,3966561721187370588,Tue Aug 18 23:37:08 +0700 2026,67943823428,imam_munthasir,Bang Bewok,-,#JakartaTerkini;#DemoMahasiswa;#bundaranhi;#be...,0,0,0,...,https://www.threads.com/@imam_munthasir/post/D...,-,-,-,2026-08-18,23:37:08,18 Agustus 2026. Aksi unjuk rasa mahasiswa UI ...,18 agustus 2026. aksi unjuk rasa mahasiswa ui ...,18 agustus 2026. aksi unjuk rasa mahasiswa ui ...,agustus aksi unjuk rasa mahasiswa ui di bundar...
4,3966561119422186685,Tue Aug 18 23:35:57 +0700 2026,76840165440,vtodaydotcom,V-Today.com,-,-,1,0,0,...,https://www.threads.com/@vtodaydotcom/post/DcM...,-,-,-,2026-08-18,23:35:57,Kritik Keras Wakil BEM UI\nFatimah Azzahra Ora...,kritik keras wakil bem ui\nfatimah azzahra ora...,kritik keras wakil bem ui fatimah azzahra oras...,kritik keras wakil bem ui fatimah azzahra oras...


In [12]:
df = df_normalize.copy()

In [18]:
# TOKENIZE
def tokenize(text):
  token = text.split()
  return token

df['tokenize'] = df['cleansing'].apply(tokenize)
df.head(5)

,post_id_str,created_at,user_id_str,username,name,user_mentions,hashtags,like_count,reply_count,repost_count,...,relation_type,target_post_id_str,target_username,tanggal,waktu,full_text,case_folding,normalize,cleansing,tokenize
0,3966562240602212698,Tue Aug 18 23:38:08 +0700 2026,69220082837,eexi,Rezky Fauziah,-,-,0,0,0,...,replied,3966468273739127284,adiitms,2026-08-18,23:38:08,Malah mikir cosplay jadi org biasa aja dulu pa...,malah mikir cosplay jadi org biasa aja dulu pa...,malah mikir cosplay jadi orang biasa saja dulu...,malah mikir cosplay jadi orang biasa saja dulu...,"[malah, mikir, cosplay, jadi, orang, biasa, sa..."
1,3966560917222668562,Tue Aug 18 23:38:08 +0700 2026,47722815717,dark_lyre,Xuě Láng,-,-,0,0,0,...,replied,3966308848940057222,koalisi.indonesia.maju,2026-08-18,23:38:08,"Kak boleh numpang tanya, itu ngapain polwan pa...","kak boleh numpang tanya, itu ngapain polwan pa...","kak boleh numpang tanya, itu mengapa polwan pa...",kak boleh numpang tanya itu mengapa polwan pak...,"[kak, boleh, numpang, tanya, itu, mengapa, pol..."
2,3966562158544928008,Tue Aug 18 23:37:58 +0700 2026,76189409752,mnbvcxzdfgjytr,.,-,-,0,0,0,...,replied,3966535614842500820,johannesclintons,2026-08-18,23:37:58,"gini ya, hampir semua gedung di thamrin dekat ...","gini ya, hampir semua gedung di thamrin dekat ...","begini ya, hampir semua gedung di thamrin deka...",begini ya hampir semua gedung di thamrin dekat...,"[begini, ya, hampir, semua, gedung, di, thamri..."
3,3966561721187370588,Tue Aug 18 23:37:08 +0700 2026,67943823428,imam_munthasir,Bang Bewok,-,#JakartaTerkini;#DemoMahasiswa;#bundaranhi;#be...,0,0,0,...,-,-,-,2026-08-18,23:37:08,18 Agustus 2026. Aksi unjuk rasa mahasiswa UI ...,18 agustus 2026. aksi unjuk rasa mahasiswa ui ...,18 agustus 2026. aksi unjuk rasa mahasiswa ui ...,agustus aksi unjuk rasa mahasiswa ui di bundar...,"[agustus, aksi, unjuk, rasa, mahasiswa, ui, di..."
4,3966561119422186685,Tue Aug 18 23:35:57 +0700 2026,76840165440,vtodaydotcom,V-Today.com,-,-,1,0,0,...,-,-,-,2026-08-18,23:35:57,Kritik Keras Wakil BEM UI\nFatimah Azzahra Ora...,kritik keras wakil bem ui\nfatimah azzahra ora...,kritik keras wakil bem ui fatimah azzahra oras...,kritik keras wakil bem ui fatimah azzahra oras...,"[kritik, keras, wakil, bem, ui, fatimah, azzah..."


In [19]:
# STOPWORD REMOVAL
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = stopwords.words('indonesian')

CUSTOM_STOPWORDS = [
    'a','b','c','d','e','f','g','h','i','j','k','l','m','n','o','p','q','r','s','t','u','v','w','x','y','z',
    'aww','awww','tu','tuh','uf','deh','dah','amp','je','rm','ufcc','ppv','of',
    'sih','eh','hm','hmm','hmmm','nya','kah','kan','kalo','kayak','doang','dll','tau',
    'ufufuf','lho','loh','lah','pon','je','pe','gs','ya','go','bro',
    'mah','lu','lo','loe','kau','ku','kuu','tau','ni','nih','gue','gua','gw',
    'kah','kahh','kak','kakk','kakak','kaka','kakkk','wkwk','wkwkwk','wkwkwkwk',
    'bang','abang','guy','guys','gaes','bawa','sok','pas','iya', 'wak'
]

stop_words.extend(CUSTOM_STOPWORDS)

sw = set(stop_words)

def stopword_removal(text):
  return [word for word in text if word not in sw]

df['stopword_removal'] = df['tokenize'].apply(stopword_removal)

df = df[df['stopword_removal'].notna() & (df['stopword_removal'].str.len() > 0)]
df.head(5)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kelil\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,post_id_str,created_at,user_id_str,username,name,user_mentions,hashtags,like_count,reply_count,repost_count,...,target_post_id_str,target_username,tanggal,waktu,full_text,case_folding,normalize,cleansing,tokenize,stopword_removal
0,3966562240602212698,Tue Aug 18 23:38:08 +0700 2026,69220082837,eexi,Rezky Fauziah,-,-,0,0,0,...,3966468273739127284,adiitms,2026-08-18,23:38:08,Malah mikir cosplay jadi org biasa aja dulu pa...,malah mikir cosplay jadi org biasa aja dulu pa...,malah mikir cosplay jadi orang biasa saja dulu...,malah mikir cosplay jadi orang biasa saja dulu...,"[malah, mikir, cosplay, jadi, orang, biasa, sa...","[mikir, cosplay, orang, pakai, baju, ala, ala,..."
1,3966560917222668562,Tue Aug 18 23:38:08 +0700 2026,47722815717,dark_lyre,Xuě Láng,-,-,0,0,0,...,3966308848940057222,koalisi.indonesia.maju,2026-08-18,23:38:08,"Kak boleh numpang tanya, itu ngapain polwan pa...","kak boleh numpang tanya, itu ngapain polwan pa...","kak boleh numpang tanya, itu mengapa polwan pa...",kak boleh numpang tanya itu mengapa polwan pak...,"[kak, boleh, numpang, tanya, itu, mengapa, pol...","[numpang, polwan, pakai, outfit, rakyat, sipil..."
2,3966562158544928008,Tue Aug 18 23:37:58 +0700 2026,76189409752,mnbvcxzdfgjytr,.,-,-,0,0,0,...,3966535614842500820,johannesclintons,2026-08-18,23:37:58,"gini ya, hampir semua gedung di thamrin dekat ...","gini ya, hampir semua gedung di thamrin dekat ...","begini ya, hampir semua gedung di thamrin deka...",begini ya hampir semua gedung di thamrin dekat...,"[begini, ya, hampir, semua, gedung, di, thamri...","[gedung, thamrin, bundaran, hi, jalan, pintas,..."
3,3966561721187370588,Tue Aug 18 23:37:08 +0700 2026,67943823428,imam_munthasir,Bang Bewok,-,#JakartaTerkini;#DemoMahasiswa;#bundaranhi;#be...,0,0,0,...,-,-,2026-08-18,23:37:08,18 Agustus 2026. Aksi unjuk rasa mahasiswa UI ...,18 agustus 2026. aksi unjuk rasa mahasiswa ui ...,18 agustus 2026. aksi unjuk rasa mahasiswa ui ...,agustus aksi unjuk rasa mahasiswa ui di bundar...,"[agustus, aksi, unjuk, rasa, mahasiswa, ui, di...","[agustus, aksi, unjuk, mahasiswa, ui, bundaran..."
4,3966561119422186685,Tue Aug 18 23:35:57 +0700 2026,76840165440,vtodaydotcom,V-Today.com,-,-,1,0,0,...,-,-,2026-08-18,23:35:57,Kritik Keras Wakil BEM UI\nFatimah Azzahra Ora...,kritik keras wakil bem ui\nfatimah azzahra ora...,kritik keras wakil bem ui fatimah azzahra oras...,kritik keras wakil bem ui fatimah azzahra oras...,"[kritik, keras, wakil, bem, ui, fatimah, azzah...","[kritik, keras, wakil, bem, ui, fatimah, azzah..."


In [20]:
# STEMMING
factory = StemmerFactory()
stemmer = factory.create_stemmer()

custom_dict = {
  "kinerja": "kinerja",
  "pemerintah": "pemerintah",
  "pemerintahan": "pemerintah",
  "kejagung": "kejagung",
  "kejaksaan": "kejaksaan",
  "kuhap": "kuhap",
  'senilai': 'nilai',
  'dinilainya': 'nilai',
  'menilainya': 'nilai',
  'bernilai': 'nilai'
}

def stemming(text):
  if isinstance(text, list):
    stemmed_words = []
    for word in text:
      if word in custom_dict:
        stemmed_words.append(custom_dict[word])
      else:
        stemmed_words.append(stemmer.stem(word))
    return stemmed_words
  else:
    return ""

df['stemming'] = df['stopword_removal'].apply(stemming)
df.head(5)

,post_id_str,created_at,user_id_str,username,name,user_mentions,hashtags,like_count,reply_count,repost_count,...,target_username,tanggal,waktu,full_text,case_folding,normalize,cleansing,tokenize,stopword_removal,stemming
0,3966562240602212698,Tue Aug 18 23:38:08 +0700 2026,69220082837,eexi,Rezky Fauziah,-,-,0,0,0,...,adiitms,2026-08-18,23:38:08,Malah mikir cosplay jadi org biasa aja dulu pa...,malah mikir cosplay jadi org biasa aja dulu pa...,malah mikir cosplay jadi orang biasa saja dulu...,malah mikir cosplay jadi orang biasa saja dulu...,"[malah, mikir, cosplay, jadi, orang, biasa, sa...","[mikir, cosplay, orang, pakai, baju, ala, ala,...","[mikir, cosplay, orang, pakai, baju, ala, ala,..."
1,3966560917222668562,Tue Aug 18 23:38:08 +0700 2026,47722815717,dark_lyre,Xuě Láng,-,-,0,0,0,...,koalisi.indonesia.maju,2026-08-18,23:38:08,"Kak boleh numpang tanya, itu ngapain polwan pa...","kak boleh numpang tanya, itu ngapain polwan pa...","kak boleh numpang tanya, itu mengapa polwan pa...",kak boleh numpang tanya itu mengapa polwan pak...,"[kak, boleh, numpang, tanya, itu, mengapa, pol...","[numpang, polwan, pakai, outfit, rakyat, sipil...","[numpang, polwan, pakai, outfit, rakyat, sipil..."
2,3966562158544928008,Tue Aug 18 23:37:58 +0700 2026,76189409752,mnbvcxzdfgjytr,.,-,-,0,0,0,...,johannesclintons,2026-08-18,23:37:58,"gini ya, hampir semua gedung di thamrin dekat ...","gini ya, hampir semua gedung di thamrin dekat ...","begini ya, hampir semua gedung di thamrin deka...",begini ya hampir semua gedung di thamrin dekat...,"[begini, ya, hampir, semua, gedung, di, thamri...","[gedung, thamrin, bundaran, hi, jalan, pintas,...","[gedung, thamrin, bundar, hi, jalan, pintas, l..."
3,3966561721187370588,Tue Aug 18 23:37:08 +0700 2026,67943823428,imam_munthasir,Bang Bewok,-,#JakartaTerkini;#DemoMahasiswa;#bundaranhi;#be...,0,0,0,...,-,2026-08-18,23:37:08,18 Agustus 2026. Aksi unjuk rasa mahasiswa UI ...,18 agustus 2026. aksi unjuk rasa mahasiswa ui ...,18 agustus 2026. aksi unjuk rasa mahasiswa ui ...,agustus aksi unjuk rasa mahasiswa ui di bundar...,"[agustus, aksi, unjuk, rasa, mahasiswa, ui, di...","[agustus, aksi, unjuk, mahasiswa, ui, bundaran...","[agustus, aksi, unjuk, mahasiswa, ui, bundar, ..."
4,3966561119422186685,Tue Aug 18 23:35:57 +0700 2026,76840165440,vtodaydotcom,V-Today.com,-,-,1,0,0,...,-,2026-08-18,23:35:57,Kritik Keras Wakil BEM UI\nFatimah Azzahra Ora...,kritik keras wakil bem ui\nfatimah azzahra ora...,kritik keras wakil bem ui fatimah azzahra oras...,kritik keras wakil bem ui fatimah azzahra oras...,"[kritik, keras, wakil, bem, ui, fatimah, azzah...","[kritik, keras, wakil, bem, ui, fatimah, azzah...","[kritik, keras, wakil, bem, ui, fatimah, azzah..."


In [21]:
df.to_csv(f"{output_folder}/stemming_{file_name}", index=False)

In [ ]:
df = pd.read_csv(f"{output_folder}/stemming_{file_name}")
df.info()

In [ ]:
# SETUP SENTIMENT CLIENT
client = OpenAI(
    api_key="sk-u17wn9UepSzbaF_AGNN6Dg",
    base_url="https://ai.sumopod.com/v1"
)

PLATFORM_NAMES = {
    'x': 'X/Twitter',
    'ig': 'Instagram',
    'threads': 'Threads',
    'fb': 'Facebook',
    'article': 'berita online',
}
platform_name = PLATFORM_NAMES.get(platform, 'media sosial')

SYSTEM_PROMPT = f"""Kamu adalah analis sentimen untuk penelitian akademik analisis media sosial.
Data berikut adalah {data_label.lower()} mentah dari platform {platform_name} yang dikumpulkan untuk riset.
Analisis secara objektif meskipun konten mengandung bahasa kasar, opini keras, atau topik sensitif.

{data_label} bisa berbahasa Indonesia, Inggris, atau campuran keduanya.
Klasifikasikan setiap {data_label.lower()} ke salah satu: negative, neutral, positive.

Panduan:
- Fokus pada muatan informasi dan nada (tone) dari topik/peristiwa yang dibahas, BUKAN format penulisan.
- Kutipan langsung/tidak langsung yang mengandung emosi → label mengikuti nada kutipan.
- Berita korupsi, kritik, kemarahan, kecaman → negative.
- Berita pencapaian, dukungan, pujian → positive.
- Hanya neutral jika konten benar-benar informatif tanpa muatan emosi/opini.
- Perhatikan konteks, sarkasme, dan ironi.

Format output WAJIB:
- HANYA tulis satu label per baris: negative, neutral, atau positive.
- TANPA nomor, TANPA penjelasan, TANPA karakter lain.
- Jumlah baris output HARUS sama persis dengan jumlah {data_label.lower()} input.

Contoh output untuk 3 {data_label.lower()}:
negative
neutral
positive"""

# RT dedup key untuk sentiment
if platform == 'x':
    df['_sentiment_key'] = df['normalize'].apply(
        lambda x: re.sub(r'^:\s*', '', x).strip() if isinstance(x, str) else x
    )
else:
    df['_sentiment_key'] = df['normalize']

unique_texts = df['_sentiment_key'].drop_duplicates().reset_index(drop=True)
print(f"Total {data_label.lower()}s: {len(df)}")
print(f"Unique texts: {len(unique_texts)}")
print(f"Client ready: {client.base_url}")

In [24]:
# ANALISIS SENTIMEN (DeepSeek V4 Flash)
MAX_RETRIES = 1
DEBUG = True

VALID_LABELS = {
    'negative', 'neutral', 'positive',
    'negatif', 'netral', 'positif'
}

LABEL_MAP = {
    'negative': 'negative', 'neutral': 'neutral', 'positive': 'positive',
    'negatif': 'negative', 'netral': 'neutral', 'positif': 'positive'
}

def parse_labels(raw_content):
    """Parse labels dari response. Handle clean, thinking text, dan label Indonesia."""
    clean_labels = []
    for line in raw_content.strip().split('\n'):
        label = line.strip().lower().lstrip('0123456789.-) ').strip().rstrip('.')
        if label in VALID_LABELS:
            clean_labels.append(LABEL_MAP[label])

    thinking_labels = []
    for match in re.finditer(r'\bsentiment\b[*\s:]*[`*]*(negative|neutral|positive|negatif|netral|positif)[`*.]*', raw_content, re.IGNORECASE):
        thinking_labels.append(LABEL_MAP[match.group(1).lower()])

    if len(clean_labels) >= len(thinking_labels):
        return clean_labels
    return thinking_labels

def _send_batch(batch):
    numbered = '\n'.join([f"{j+1}. {t}" for j, t in enumerate(batch)])
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.chat.completions.create(
                model="deepseek-v4-flash",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": numbered}
                ],
                temperature=0,
                max_tokens=16000
            )
            raw = response.choices[0].message.content or ''
            labels = parse_labels(raw)
            if len(labels) == len(batch):
                return labels
            if DEBUG:
                print(f"    attempt {attempt}: parsed {len(labels)}/{len(batch)}")
                print(f"    raw ({len(raw)} chars): '{raw[:300]}'")
        except Exception as e:
            if DEBUG:
                print(f"    attempt {attempt} error: {e}")
        if attempt < MAX_RETRIES:
            time.sleep(5 * attempt)
    return None

def predict_sentiment(texts, batch_size=50):
    all_preds = [None] * len(texts)
    pending = list(range(len(texts)))

    for current_bs in [batch_size, 10, 1]:
        if not pending:
            break
        if current_bs < batch_size:
            print(f"\n[RETRY] {len(pending)} {data_label.lower()}s dengan batch_size={current_bs}")

        next_pending = []
        for i in tqdm(range(0, len(pending), current_bs),
                      desc=f"bs={current_bs}", disable=len(pending) < current_bs):
            idx_chunk = pending[i:i+current_bs]
            batch = [texts[idx] for idx in idx_chunk]
            result = _send_batch(batch)

            if result:
                for idx, label in zip(idx_chunk, result):
                    all_preds[idx] = label
            else:
                if DEBUG and current_bs > 1:
                    print(f"  [FAILED] batch starting idx {idx_chunk[0]} (bs={current_bs})")
                next_pending.extend(idx_chunk)

            time.sleep(0.3)

        pending = next_pending

    if pending:
        print(f"\n[WARNING] {len(pending)} {data_label.lower()}s tetap NaN setelah semua retry")
    else:
        print(f"\n[OK] Semua {len(texts)} {data_label.lower()}s berhasil diprediksi")

    return all_preds

# Cek apakah sudah ada kolom sentiment (re-run NaN only)
if 'sentiment' in df.columns and df['sentiment'].notna().any():
    nan_keys = df[df['sentiment'].isna()]['_sentiment_key'].drop_duplicates().reset_index(drop=True)
    print(f"Re-run mode: {len(nan_keys)} NaN unique texts (skip {len(unique_texts) - len(nan_keys)} yang sudah ada)")

    if len(nan_keys) > 0:
        nan_preds = predict_sentiment(nan_keys.tolist())
        nan_map = dict(zip(nan_keys, nan_preds))
        mask = df['sentiment'].isna()
        df.loc[mask, 'sentiment'] = df.loc[mask, '_sentiment_key'].map(nan_map)
    else:
        print("[OK] Tidak ada NaN — skip predict")
else:
    print(f"First run: {len(unique_texts)} unique texts")
    unique_preds = predict_sentiment(unique_texts.tolist())
    sentiment_map = dict(zip(unique_texts, unique_preds))
    df['sentiment'] = df['_sentiment_key'].map(sentiment_map)

df.drop(columns=['_sentiment_key'], inplace=True, errors='ignore')

nan_count = df['sentiment'].isna().sum()
print(f"\nHasil: {len(df)} {data_label.lower()}s")
print(f"NaN sentiment: {nan_count}")
print(df['sentiment'].value_counts().to_string())

First run: 10313 unique texts


bs=50:   0%|          | 0/207 [00:00<?, ?it/s]

bs=50:   1%|          | 2/207 [01:40<2:45:25, 48.42s/it]

    attempt 1: parsed 53/50
    raw (467 chars): 'negative
negative
negative
negative
negative
negative
positive
negative
negative
negative
positive
negative
negative
negative
negative
negative
negative
negative
neutral
negative
negative
positive
neutral
neutral
neutral
negative
negative
positive
positive
positive
positive
neutral
negative
negative'
  [FAILED] batch starting idx 100 (bs=50)


bs=50:   6%|▌         | 12/207 [47:04<25:03:16, 462.55s/it]

    attempt 1: parsed 49/50
    raw (428 chars): 'negative
positive
positive
positive
neutral
positive
positive
neutral
positive
negative
positive
neutral
negative
positive
neutral
positive
positive
negative
neutral
neutral
positive
neutral
positive
neutral
positive
neutral
positive
positive
neutral
positive
neutral
positive
positive
positive
posit'
  [FAILED] batch starting idx 600 (bs=50)


... (116 outputs left out)

bs=10: 100%|██████████| 290/290 [2:45:55<00:00, 34.33s/it]   



[RETRY] 10 posts dengan batch_size=1


bs=1: 100%|██████████| 10/10 [00:39<00:00,  3.95s/it]



[OK] Semua 10313 posts berhasil diprediksi

Hasil: 11107 posts
NaN sentiment: 0
sentiment
positive    4666
negative    3540
neutral     2901


In [ ]:
# # TEST API: 1 tweet saja
# client = OpenAI(
#     api_key="sk-u17wn9UepSzbaF_AGNN6Dg",
#     base_url="https://ai.sumopod.com/v1"
# )

# test_response = client.chat.completions.create(
#     model="deepseek-v4-flash",
#     messages=[
#         {"role": "system", "content": "Classify sentiment: negative, neutral, or positive. Output only the label."},
#         {"role": "user", "content": "1. saya senang hari ini cerah sekali"}
#     ],
#     temperature=0,
#     max_tokens=100
# )

# print(f"Model: {test_response.model}")
# print(f"Finish reason: {test_response.choices[0].finish_reason}")
# print(f"Content: '{test_response.choices[0].message.content}'")
# print(f"Usage: {test_response.usage}")

Model: deepseek-v4-flash
Finish reason: stop
Content: 'positive'
Usage: CompletionUsage(completion_tokens=43, prompt_tokens=109, total_tokens=152, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=41, rejected_prediction_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cache_write_tokens=None, cached_tokens=0), prompt_cache_hit_tokens=0, prompt_cache_miss_tokens=109)


In [ ]:
# DEBUG: Cek isi tweet yang selalu NaN + test batch kecil
nan_keys = df[df['sentiment'].isna()]['_sentiment_key'].drop_duplicates().reset_index(drop=True).tolist()
print(f"Total NaN: {len(nan_keys)}")
print(f"\n--- Sample 5 tweet NaN ---")
for i, t in enumerate(nan_keys[:5]):
    print(f"{i+1}. {t[:100]}")

# Test batch kecil (10 tweets) untuk lihat response
batch = nan_keys[:10]
numbered = '\n'.join([f"{j+1}. {t}" for j, t in enumerate(batch)])

response = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": numbered}
    ],
    temperature=0,
    max_tokens=4000
)

raw = response.choices[0].message.content or ''
print(f"\n--- Response (batch 10) ---")
print(f"Length: {len(raw)} chars")
print(f"Content: '{raw[:500]}'")
print(f"\nFinish reason: {response.choices[0].finish_reason}")
print(f"Model: {response.model}")

In [ ]:
# DISTRIBUSI SENTIMEN BARCHART
color_map = {
    "negative": "#F44336",
    "neutral": "#FFC107",
    "positive": "#4CAF50"
}

sentiment_counts = df["sentiment"].value_counts()

colors = [color_map[label] for label in sentiment_counts.index]

plt.figure(figsize=(7,4))
bars = sentiment_counts.plot(
    kind="bar",
    color=colors,
    alpha=0.8
)

for i, v in enumerate(sentiment_counts):
    pct = (v / len(df['sentiment'])) * 100
    plt.text(i, v, f"{v} | {pct:.2f}%", ha="center", va="bottom", fontsize=10)

plt.title(f"Distribusi Sentimen \"{project_name}\" (LLM)", fontsize=14, fontweight="bold")
plt.xlabel("Kelas Sentimen", fontsize=12)
plt.ylabel(f"Jumlah {data_label}", fontsize=12)
plt.xticks(rotation=0)
plt.ylim(0, max(sentiment_counts) * 1.1)
plt.grid(axis="y", linestyle="--", alpha=0.6)

plt.savefig(f"{output_folder}/{project_name}_barchart_sentiment.png", dpi=300, bbox_inches="tight")
plt.show()

In [25]:
# SIMPAN HASIL
df.to_csv(f"{output_folder}/prep_{file_name}", index=False, encoding="utf-8")